# Wrapper Feature Selection – Reusable Template

**Short name:** `WrapFS_Obes`  
Drop in any numeric table with a binary label. Wrappers wrap *this* estimator — change `EST` if you switch models.

```
load → baseline LR → SFS / SBS (mlxtend or sklearn) → scale → RFE → compare k → hold-out check
```


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.model_selection import train_test_split
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs

# --- edit ---
CSV = "data/obesity.csv"
LABEL = "NObeyesdad"
K_SFS = 9
K_SBS = 7
K_RFE = 8
TEST_SIZE = 0.30
SEED = 0
# ------------

df = pd.read_csv(CSV)
y = df[LABEL]
X = df.drop(columns=[LABEL])
EST = LogisticRegression(max_iter=1000)
print("n", len(df), "p", X.shape[1], "base", EST.fit(X, y).score(X, y))

sfs = SFS(EST, k_features=K_SFS, forward=True, floating=False, scoring="accuracy", cv=0).fit(X, y)
print("SFS", sfs.subsets_[K_SFS]["feature_names"], sfs.subsets_[K_SFS]["avg_score"])

sbs = SFS(EST, k_features=K_SBS, forward=False, floating=False, scoring="accuracy", cv=0).fit(X, y)
print("SBS", sbs.subsets_[K_SBS]["feature_names"], sbs.subsets_[K_SBS]["avg_score"])

feats = list(X.columns)
Xs = pd.DataFrame(StandardScaler().fit_transform(X), columns=feats)
rfe = RFE(EST, n_features_to_select=K_RFE).fit(Xs, y)
print("RFE", [f for f, ok in zip(feats, rfe.support_) if ok], rfe.score(Xs, y))

plot_sfs(sfs.get_metric_dict()); plt.title("SFS"); plt.show()

cols = list(sfs.subsets_[K_SFS]["feature_names"])
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
print("hold-out all", EST.fit(Xtr, ytr).score(Xte, yte),
      "hold-out SFS", EST.fit(Xtr[cols], ytr).score(Xte[cols], yte))
